In [4]:
from typing import Annotated,Sequence,TypedDict
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage,ToolMessage,SystemMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode
load_dotenv()

True

In [6]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    

In [7]:
@tool
def add(a:int,b:int):
    """this is a tool to add two numbers"""
    return a + b
tools=[add]
model = ChatOpenAI(model="gpt-5-nano").bind_tools(tools)


In [9]:
def model_call(state:AgentState)->AgentState:
    system_prompt=SystemMessage(content="You are my ai Assistant, please answer the questions best on your ability")
    response=model.invoke([system_prompt]+state["messages"])

    return {"messages":[response]}

In [10]:
def should_continue(state:AgentState):
    messages =state["messages"]
    last_message = messages[-1]
    if not last_message.tool_calls:
        return "end"
    else:
        return "continue"
    

In [12]:
graph=StateGraph(AgentState)
graph.add_node("model_call",model_call)
tool_node=ToolNode(tools)
graph.add_node(tool_node)
graph.add_edge(START,"model_call")
graph.add_conditional_edges("model_call",should_continue,{"continue":"tool_node","end":END})
graph.add_edge("tool_node","model_call")
app=graph.compile()


ValueError: Found edge starting at unknown node 'tool_node'